In [1]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine

# ==========================================
# Connection & Environment Configuration
# ==========================================
SERVER_NAME = r'ALEXANDER\SQLEXPRESS'
DATABASE_NAME = 'SupplyChainDW'
SCHEMA_NAME = 'Bronze'
BASE_PATH = r'C:\Users\BLESSKID\OneDrive\Documents\self_practice\datasets\supply_chain_dataset'

# Explicit file-to-table mappings
FILE_MAPPINGS = [
    {
        "file": "DescriptionDataCoSupplyChain.csv",
        "table": "DataCoDescriptionRaw"
    },
    {
        "file": "DataCoSupplyChainDataset.csv",
        "table": "DataCoSupplyChainRaw"
    },
    {
        "file": "tokenized_access_logs.csv",
        "table": "DataCoTokenRaw"
    }
]

# ==========================================
# High-Speed Engine Setup
# ==========================================
connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

quoted_conn = urllib.parse.quote_plus(connection_string)

# fast_executemany=True dramatically speeds up multi-row inserts in pyodbc
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={quoted_conn}",
    fast_executemany=True
)

# ==========================================
# Ingestion Execution Loop
# ==========================================
for item in FILE_MAPPINGS:
    file_name = item["file"]
    target_table = item["table"]
    full_path = os.path.join(BASE_PATH, file_name)
    
    if os.path.exists(full_path):
        print(f"📖 Reading: {file_name}...")
        
        # Read raw CSV using latin1 to safely handle non-ASCII characters
        df = pd.read_csv(full_path, encoding='latin1', low_memory=False)
        
        # Cast to string to ensure resilient Bronze staging without truncation errors
        df = df.astype(str)
        
        print(f"🚀 Ingesting {len(df):,} rows into [{SCHEMA_NAME}].[{target_table}]...")
        
        df.to_sql(
            name=target_table,
            con=engine,
            schema=SCHEMA_NAME,
            if_exists='replace',  # Recreates table for clean Bronze staging
            index=False,
            chunksize=5000
        )
        
        print(f"✅ Successfully loaded [{SCHEMA_NAME}].[{target_table}]\n")
    else:
        print(f"❌ Error: Could not locate file at '{full_path}'\n")

print("🎉 Ingestion Complete! All 3 raw datasets are populated in SQL Server.")

📖 Reading: DescriptionDataCoSupplyChain.csv...
🚀 Ingesting 52 rows into [Bronze].[DataCoDescriptionRaw]...
✅ Successfully loaded [Bronze].[DataCoDescriptionRaw]

📖 Reading: DataCoSupplyChainDataset.csv...
🚀 Ingesting 180,519 rows into [Bronze].[DataCoSupplyChainRaw]...
✅ Successfully loaded [Bronze].[DataCoSupplyChainRaw]

📖 Reading: tokenized_access_logs.csv...
🚀 Ingesting 469,977 rows into [Bronze].[DataCoTokenRaw]...
✅ Successfully loaded [Bronze].[DataCoTokenRaw]

🎉 Ingestion Complete! All 3 raw datasets are populated in SQL Server.
